In [1]:
# ═══════════════════════════════════════════════════════════════════════
# MAPLE SHIELD — WSA CLAIMS ADJUDICATOR
# Phase 1: Setup & Environment Test
# ═══════════════════════════════════════════════════════════════════════
# Paste each CELL block into a separate Colab cell and run in order.
# Do not proceed to Phase 2 until all three cells run without errors.
# ═══════════════════════════════════════════════════════════════════════


# ───────────────────────────────────────────────────────────────────────
# CELL 1-A │ Install packages
# ───────────────────────────────────────────────────────────────────────
# Paste this into your first Colab cell and run it.
# Expected output: Successfully installed ... (no red errors)

In [2]:
!pip install -q google-generativeai Pillow pymupdf python-docx pydantic python-docx pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 27.0 MB/s eta 0:00:00


In [3]:
!pip install -q gradio==6.19.0

In [4]:
!pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 15.0 MB/s eta 0:00:00


In [5]:
# ───────────────────────────────────────────────────────────────────────
# CELL 1-B │ Imports & API configuration
# ───────────────────────────────────────────────────────────────────────
# Paste into second Colab cell and run.
# Expected output: ✓ All imports OK

In [6]:
import gradio as gr
import json
import re
import os
import copy
import fitz                          # PyMuPDF — PDF parsing
from datetime import datetime
from pathlib import Path
from PIL import Image
import anthropic
from google.colab import userdata

# Initialize Anthropic Client
client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

# Model Mapping
SONNET = 'claude-sonnet-4-6'           #"claude-4-6-sonnet-20240620"
HAIKU  = 'claude-haiku-4-5-20251001'   #"claude-4-5-haiku-20240307"
OPUS   = 'claude-opus-4-8'             #"claude-4-8-opus-20240229"

print("✓ Anthropic Client Initialized")

✓ Anthropic Client Initialized


In [7]:
import os

# Define the specific project folder path
PROJECT_FOLDER = '/content/drive/MyDrive/AI_Builder_Adjudication_Agent_Case_2026/wellness-claims-agentic-pipeline'

def verify_workspace():
    if os.path.exists(PROJECT_FOLDER):
        print(f"✓ Workspace found: {PROJECT_FOLDER}")
        print("Files available:")
        for file in os.listdir(PROJECT_FOLDER):
            print(f"  - {file}")
    else:
        print(f"✗ Error: Folder not found at {PROJECT_FOLDER}")

verify_workspace()

✓ Workspace found: /content/drive/MyDrive/AI_Builder_Adjudication_Agent_Case_2026/wellness-claims-agentic-pipeline
Files available:
  - prompts
  - configs
  - agents
  - test_set
  - .ipynb_checkpoints
  - requirements.txt
  - __init__.py
  - create_agent_prompts.py
  - __pycache__
  - app.py
  - pipeline.py


In [8]:
import sys
import os

# Clean up sys.modules to ensure a fresh import
# Important: delete the most specific entries first
if 'agents.schemas' in sys.modules:
    del sys.modules['agents.schemas']
    print("✓ Removed 'agents.schemas' from sys.modules (if present)")
if 'agents' in sys.modules:
    # Safely clear sub-modules related to 'agents' if they exist
    for mod in list(sys.modules.keys()):
        if mod.startswith('agents.'):
            del sys.modules[mod]
    # Then remove the 'agents' package itself if it was loaded as a package
    if 'agents' in sys.modules and hasattr(sys.modules['agents'], '__path__'):
        del sys.modules['agents']
    print("✓ Cleared 'agents' related modules from sys.modules (if present)")


# Ensure PROJECT_FOLDER is at the beginning of sys.path to prioritize local modules
# This is critical for Python to find `agents/schemas.py` correctly.
if PROJECT_FOLDER in sys.path:
    sys.path.remove(PROJECT_FOLDER)
sys.path.insert(0, PROJECT_FOLDER)
print(f"✓ PROJECT_FOLDER '{PROJECT_FOLDER}' ensured at start of sys.path")

# The original content also ran the schemas.py file directly. If its output is important,
# it can be re-added, but it's not strictly necessary for the 'from agents.schemas import ClaimRecord' to work.
# For now, focus on the import issue directly.

from agents.schemas import ClaimRecord
print('✓ ClaimRecord imported successfully within Colab kernel')
print('ok')

✓ PROJECT_FOLDER '/content/drive/MyDrive/AI_Builder_Adjudication_Agent_Case_2026/wellness-claims-agentic-pipeline' ensured at start of sys.path
✓ ClaimRecord imported successfully within Colab kernel
ok


In [9]:
# import os
# from google.colab import userdata

# # Retrieve the API key from Colab secrets
# ANTHROPIC_API_KEY_VALUE = userdata.get('ANTHROPIC_API_KEY')

# # Run the Python script, explicitly setting ANTHROPIC_API_KEY as an environment variable
# # for the subprocess. We also keep PYTHONPATH to ensure modules are found.
# !ANTHROPIC_API_KEY="{ANTHROPIC_API_KEY_VALUE}" PYTHONPATH="{PROJECT_FOLDER}" python "{PROJECT_FOLDER}/agents/models/extractor.py"

Extractor: Read prompt file successfully.
Testing file parser (no API call)...

  ✓ 01_climbing_gym_receipt.jpg              → image_b64    (29 KB)
  ✓ 02_meditation_app.jpg                    → image_b64    (61 KB)
  ✓ 03_grocery_receipt.jpg                   → image_b64    (77 KB)
  ✓ 04_trainer_invoice.pdf                   → image_b64    (110 KB)
  ✓ 05_office_chair.jpg                      → image_b64    (68 KB)
  ✓ 06_season_pass_email.txt                 → text         (0 KB)
  ✓ 07_gym_membership.jpg                    → image_b64    (72 KB)
  ✓ 08_home_gym_quote.pdf                    → image_b64    (124 KB)
  ✓ 09_physio_clinic_invoice.pdf             → image_b64    (142 KB)
  ✓ 10_course_enrollment_email.txt           → text         (0 KB)
  ✓ 11_gym_portal_receipt.jpg                → image_b64    (62 KB)
  ✓ 12_yoga_classpack.jpg                    → image_b64    (62 KB)
  ✓ 13_childcare_invoice.pdf                 → image_b64    (116 KB)
  ✓ 14_running_shoes.jpg          

In [ ]:
# # Retrieve the API key from Colab secrets
# ANTHROPIC_API_KEY_VALUE = userdata.get('ANTHROPIC_API_KEY')

# # Run the Python script, explicitly setting ANTHROPIC_API_KEY as an environment variable
# # for the subprocess. We also keep PYTHONPATH to ensure modules are found.
# # !ANTHROPIC_API_KEY="{ANTHROPIC_API_KEY_VALUE}" PYTHONPATH="{PROJECT_FOLDER}" python "{PROJECT_FOLDER}/agents/models/extractor.py"

# !ANTHROPIC_API_KEY="{ANTHROPIC_API_KEY_VALUE}" PYTHONPATH="{PROJECT_FOLDER}" python "{PROJECT_FOLDER}/agents/models/validator.py"

Testing validator...

Test 1 — GoodLife Fitness gym membership (expect PASS)
  overall_gate_status    : PASS
  category_code          : WSA-FIT
  gate1 passed           : True
  gate2_expense passed   : True
  gate2_period passed    : True
  gate3 passed           : True
  balance sufficient     : True
  first_failing_gate     : None

Test 2 — 2023 season pass (expect FAIL — out of period)
  overall_gate_status    : FAIL
  gate2_period passed    : False
  gate2_period finding   : The expense was incurred on 2023-11-14, which falls in the 2023 calendar year. The benefit year for this plan is January 1, 2025 through December 31, 2025. An expense incurred in 2023 cannot be claimed against the 2025 benefit year, regardless of when it was submitted. Gate 2b fails.
  first_failing_gate     : gate2_period

Test 3 — Duplicate check (script — no API call)
  is_duplicate: True  (expect True)
  duplicate_of: 007  (expect 007)

  ✓ Serialises correctly

✓ Phase 4 complete — proceed to Phase 5 (adj

In [ ]:
# # Retrieve the API key from Colab secrets
# ANTHROPIC_API_KEY_VALUE = userdata.get('ANTHROPIC_API_KEY')

# # Run the Python script, explicitly setting ANTHROPIC_API_KEY as an environment variable
# # for the subprocess. We also keep PYTHONPATH to ensure modules are found.
# # !ANTHROPIC_API_KEY="{ANTHROPIC_API_KEY_VALUE}" PYTHONPATH="{PROJECT_FOLDER}" python "{PROJECT_FOLDER}/agents/models/extractor.py"

# !ANTHROPIC_API_KEY="{ANTHROPIC_API_KEY_VALUE}" PYTHONPATH="{PROJECT_FOLDER}" python "{PROJECT_FOLDER}/agents/models/adjudicator.py"

Adjudicator: Read prompt file successfully.
Testing adjudicator...

Test 1 — GoodLife Fitness (expect APPROVE, confidence >= 0.80)
  decision             : APPROVE
  reason_code          : A-OK
  approved_amount      : 169.5
  confidence_score     : 0.9
  tax_flag             : True
  triage_tag           : CONFIDENT
  rationale            : This looks like a clean, eligible claim — I'd suggest approving it in full. All four gates passed: Jordan Tran is an eli...
  confidence_reasoning : Starting at 1.0. Image quality good: no deduction. No unreadable fields: no deduction. Date present and within period: n...

Test 2 — 2023 season pass (expect DENY D-PER)
  decision     : DENY
  reason_code  : D-PER
  confidence   : 0.9
  triage_tag   : CONFIDENT
  rationale    : This one's a clear period failure. The expense is a perfectly eligible season ski pass (WSA-FIT) for an eligible person ...

Test 3 — Physiotherapy (expect ROUTE_HCSA)
  decision     : ROUTE_HCSA
  reason_code  : R-HCSA
  conf

In [ ]:
# # Retrieve the API key from Colab secrets
# ANTHROPIC_API_KEY_VALUE = userdata.get('ANTHROPIC_API_KEY')

# # Run the Python script, explicitly setting ANTHROPIC_API_KEY as an environment variable
# # for the subprocess. We also keep PYTHONPATH to ensure modules are found.
# # !ANTHROPIC_API_KEY="{ANTHROPIC_API_KEY_VALUE}" PYTHONPATH="{PROJECT_FOLDER}" python "{PROJECT_FOLDER}/agents/models/extractor.py"

# !ANTHROPIC_API_KEY="{ANTHROPIC_API_KEY_VALUE}" PYTHONPATH="{PROJECT_FOLDER}" python "{PROJECT_FOLDER}/pipeline.py"

Extractor: Read prompt file successfully.
Validator: Read prompt file successfully.
Adjudicator: Read prompt file successfully.
  Loaded 4 feedback entries from feedback_log.json
Pipeline integration test


═══════════════════════════════════════════════════════
  Processing claim #001  —  07_gym_membership.jpg
═══════════════════════════════════════════════════════
  [1/5] Extractor (Haiku)  ...
        vendor : GOODLIFE FITNESS
        amount : 169.5
        date   : 2025-03-14
        quality: good
  [2/5] Duplicate check ...
        ✓  No duplicate found
  [3/5] Balance: $750.00 remaining
  [4/5] Validator (Sonnet) ...
        gate status : PASS
        category    : WSA-FIT
  [5/5] Adjudicator (Opus) ...
        decision   : APPROVE (A-OK)
        confidence : 90%
        triage     : CONFIDENT
  ✓ Claim #001 stored — awaiting examiner review

  Result: APPROVE (A-OK)
  Confidence: 90%
  Triage: CONFIDENT
  Rationale: This looks like a clean, eligible claim — I'd suggest approving

In [ ]:
# Retrieve the API key from Colab secrets
ANTHROPIC_API_KEY_VALUE = userdata.get('ANTHROPIC_API_KEY')

# Run the Python script, explicitly setting ANTHROPIC_API_KEY as an environment variable
# for the subprocess. We also keep PYTHONPATH to ensure modules are found.
# !ANTHROPIC_API_KEY="{ANTHROPIC_API_KEY_VALUE}" PYTHONPATH="{PROJECT_FOLDER}" python "{PROJECT_FOLDER}/agents/models/extractor.py"

!ANTHROPIC_API_KEY="{ANTHROPIC_API_KEY_VALUE}" PYTHONPATH="{PROJECT_FOLDER}" python "{PROJECT_FOLDER}/app.py"

Extractor: Read prompt file successfully.
Validator: Read prompt file successfully.
Adjudicator: Read prompt file successfully.
/content/drive/MyDrive/AI_Builder_Adjudication_Agent_Case_2026/wellness-claims-agentic-pipeline/app.py:335: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://87a708d947d31edbbf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_hel

In [ ]:
# !cat "{PROJECT_FOLDER}/prompts/extractor_system.txt" | head -5

You are the Extraction Agent for the Maple Shield WSA Claims Adjudication Pipeline.

Your only job is to read a receipt and extract structured facts from it.
You do not apply any rules. You do not make any decisions. You do not judge eligibility.
You read what is there and report exactly what you see.
